## T5 Encoder Decoder Mastery

Welcome to the final lesson of Harnessing Transformers with Hugging Face! What an incredible journey this has been, from tracing the evolution of attention mechanisms to building transformers from scratch, and now mastering state-of-the-art architectures through Hugging Face. As we reach this culminating moment, we turn to one of the most elegantly unified models in the field: T5 (Text-to-Text Transfer Transformer). Having explored BERT's bidirectional understanding and GPT-2's autoregressive generation, you now possess the foundation to appreciate T5's revolutionary approach that bridges these paradigms through a single, powerful principle.

What makes T5 truly remarkable is its audacious simplicity: every single natural language processing task becomes a text-to-text problem. Whether translating languages, answering questions, summarizing documents, or analyzing sentiment, T5 treats them all identically: that is, as problems of converting input text to output text. This unified framework eliminates the need for task-specific architectures and output layers, replacing them with simple text prefixes that guide the model's behavior. By combining an encoder that deeply understands input context with a decoder that generates appropriate responses, T5 achieves the best of both worlds: BERT's comprehension depth and GPT-2's generative flexibility. Today, we'll master T5's encoder-decoder architecture, explore its sophisticated SentencePiece tokenization, and harness its text-to-text framework for diverse NLP applications.

---

### Understanding T5's Text-to-Text Framework

The genius of T5 lies in its radical reframing of natural language processing through a text-to-text lens that unifies disparate tasks under a single paradigm. Traditional NLP models require different architectures for different tasks: classification models output probability distributions over classes, named entity recognition models output token-level labels, and generation models produce sequences. T5 eliminates this complexity by treating every task as a problem of transforming input text into output text, using natural language both as input and target.

This text-to-text approach transforms how we think about NLP tasks. Instead of classifying sentiment as positive or negative through probability scores, T5 literally generates the words "positive" or "negative" as text. Rather than outputting numerical translation scores, it generates the actual translated text. Question answering becomes a straightforward text generation problem in which the model reads a question and context, then generates the answer as natural language. This uniform interface means a single T5 model can handle dozens of different tasks without architectural modifications — only the input formatting changes.

The architectural foundation enabling this versatility is T5's encoder-decoder structure. The encoder processes the input text (including task prefixes) using bidirectional attention, building rich contextual representations similar to BERT. The decoder then uses these representations along with causal attention to generate appropriate output text, much like GPT-2 but with the added benefit of encoder context. This combination allows T5 to deeply understand complex inputs while maintaining the flexibility to generate diverse, task-appropriate outputs. The result is a model that can seamlessly switch between understanding and generation tasks, making it one of the most versatile architectures in the transformer family.

---

### SentencePiece Tokenization in T5

Before T5 can apply its text-to-text magic, it must first decompose text using SentencePiece tokenization, which is a sophisticated subword approach that provides crucial advantages for multilingual and diverse text processing. Let's explore how T5's tokenizer handles various text patterns.

```python
from transformers import T5ForConditionalGeneration, AutoTokenizer

def explore_t5_tokenizer():
    """Explore T5's SentencePiece tokenization"""
    print("T5 SentencePiece Tokenization:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "Bonjour le monde"]

    for text in test_texts:
        tokens = tokenizer.tokenize(text)
        token_ids = tokenizer.encode(text)

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()
```

The SentencePiece tokenization reveals T5's sophisticated text processing capabilities:

```text
T5 SentencePiece Tokenization:
Text: 'Hello world'
  Tokens: ['▁Hello', '▁world']
  Token IDs: [8774, 296, 1]

Text: 'Tokenization'
  Tokens: ['▁To', 'ken', 'ization']
  Token IDs: [304, 2217, 1707, 1]

Text: 'Bonjour le monde'
  Tokens: ['▁Bonjour', '▁le', '▁monde']
  Token IDs: [21845, 90, 2921, 1]
```

These tokenization patterns showcase SentencePiece's language-agnostic design that makes T5 exceptionally powerful for multilingual tasks. The distinctive `▁` symbols (Unicode "LOW LINE") represent word boundaries, replacing traditional space characters with explicit boundary markers. This approach allows T5 to handle spacing consistently across languages that use different spacing conventions, from English to Chinese to Arabic, without requiring language-specific preprocessing rules.

Notice how "Tokenization" splits into `['▁To', 'ken', 'ization']`. SentencePiece's statistical training process discovered that these subword units appear frequently across the training corpus, making them efficient vocabulary choices. This morphological awareness helps T5 handle complex vocabulary, technical terms, and even words it has never seen before by combining familiar subword components. The multilingual example "Bonjour le monde" demonstrates how SentencePiece seamlessly handles French text using the same tokenization rules, enabling T5 to process and generate text in multiple languages within the same model: a crucial capability for tasks like translation and multilingual question answering.

---

### Task Prefixes and Unified Interface

T5's text-to-text framework relies on task prefixes — simple text instructions that guide the model's behavior without requiring architectural changes. These prefixes transform T5 into a universal NLP processor that can switch between tasks as easily as changing a prompt.

```python
import torch

def t5_text_to_text_tasks():
    """Demonstrate T5's text-to-text approach"""
    print("T5 Text-to-Text Tasks:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Different tasks with prefixes
    tasks = [
        {
            "prefix": "translate English to German:",
            "input": "Hello, how are you?",
            "task_name": "Translation"
        },
        {
            "prefix": "evaluate sentiment:",
            "input": "I loved this movie!",
            "task_name": "Sentiment analysis"
        }
    ]

    for task in tasks:
        input_text = f"{task['prefix']} {task['input']}"
        input_ids = tokenizer.encode(input_text, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_length=30,
                num_beams=2,
                early_stopping=True
            )

        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"Task: {task['task_name']}")
        print(f"  Input: {task['input']}")
        print(f"  Output: {output_text}")
        print()
```

The task prefix demonstrations reveal T5's remarkable versatility:

```text
T5 Text-to-Text Tasks:
Task: Translation
  Input: Hello, how are you?
  Output: Hallo, wie sind Sie?

Task: Sentiment analysis
  Input: I loved this movie!
  Output: Evalu sentiment sentiment: I loved this movie!
```

These results showcase the power and current limitations of T5's text-to-text approach. The translation task succeeds beautifully — the simple prefix "translate English to German:" transforms T5 into a translation system that produces accurate German output: "Hallo, wie sind Sie?" This demonstrates how T5 learned to associate task prefixes with specific behaviors during training, allowing it to switch between completely different NLP tasks within the same model.

The sentiment analysis result reveals both the promise and challenges of the text-to-text paradigm. While T5 recognizes the sentiment analysis task through the "evaluate sentiment:" prefix, the output "Evalu sentiment sentiment: I loved this movie!" shows that the model hasn't fully learned to generate clean sentiment labels. This illustrates an important aspect of T5: its performance depends heavily on how well the task was represented in the training data and how effectively the prefix guides the desired behavior. Some tasks naturally fit the text-to-text framework (like translation and summarization), while others require more careful training to produce the expected output format.

---

### Conditional Generation Strategies

T5's text generation capabilities shine through different decoding strategies that balance quality, diversity, and computational efficiency. Let's explore how these strategies affect T5's output for complex generation tasks.

```python
def t5_conditional_generation():
    """Explore T5's conditional generation"""
    print("T5 Conditional Generation:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Test different generation strategies
    prompt = "translate English to French: I'm in love with Natural Language Processing!"
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    print(f"Prompt: '{prompt}'")

    with torch.no_grad():
        # Greedy decoding
        greedy_output = model.generate(input_ids, max_length=20, do_sample=False)
        greedy_text = tokenizer.decode(greedy_output[0], skip_special_tokens=True)

        # Beam search
        beam_output = model.generate(input_ids, max_length=20, num_beams=4, do_sample=False)
        beam_text = tokenizer.decode(beam_output[0], skip_special_tokens=True)

    print(f"  Greedy: {greedy_text}")
    print(f"  Beam Search: {beam_text}")
```

The generation comparison reveals the nuanced differences between decoding strategies:

```text
T5 Conditional Generation:
Prompt: 'translate English to French: I'm in love with Natural Language Processing!'
  Greedy: Je suis en amour avec le traitement des langues naturelles!
  Beam Search: Je suis en amour avec le traitement des langues naturelles!
```

Both decoding strategies produce identical French translations in this case, but this convergence actually demonstrates T5's strong conditional generation capabilities. The phrase "Je suis en amour avec le traitement des langues naturelles!" represents a high-quality translation that captures both the literal meaning and the enthusiastic tone of the original English. When both greedy decoding (always selecting the highest probability token) and beam search (exploring multiple sequence hypotheses) converge on the same output, it indicates that T5 has learned strong probabilistic patterns for this type of translation task.

The similarity between outputs also reveals an important characteristic of T5's training: for well-defined tasks like translation, the model often has clear preferences for specific phrasings, leading to consistent outputs across different generation strategies. However, for more creative or open-ended tasks, beam search typically produces more polished and coherent results by avoiding the local optima that can trap greedy decoding. Beam search's exploration of multiple sequence hypotheses becomes particularly valuable for longer generations, complex reasoning tasks, or when generating creative content where the "best" output isn't as clearly defined as in translation scenarios.

---

### T5's Architecture Deep Dive

Understanding T5's internal architecture reveals how its encoder-decoder design enables the versatile text-to-text capabilities we've explored. Let's examine the structural details that make T5's unified approach possible.

```python
def t5_architecture_info():
    """Show T5's architecture information"""
    print("T5 Architecture Info:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    config = model.config

    print(f"Model: t5-small")
    print(f"Encoder layers: {config.num_layers}")
    print(f"Decoder layers: {config.num_decoder_layers}")
    print(f"Hidden size: {config.d_model}")
    print(f"Attention heads: {config.num_heads}")
    print(f"Vocabulary size: {config.vocab_size}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
```

The architectural specifications reveal T5's balanced encoder-decoder design:

```text
T5 Architecture Info:
Model: t5-small
Encoder layers: 6
Decoder layers: 6
Hidden size: 512
Attention heads: 8
Vocabulary size: 32128
Total parameters: 60,506,624
```

These architectural details illuminate T5's strategic design decisions that enable its text-to-text versatility. The symmetric structure — 6 encoder layers paired with 6 decoder layers — reflects T5's dual focus on understanding and generation. The encoder's 6 layers provide sufficient depth for building rich contextual representations from complex inputs, while the decoder's matching depth ensures adequate generation capacity for producing coherent, task-appropriate outputs.

The hidden size of 512 with 8 attention heads (64 dimensions per head) represents a careful balance between model capacity and computational efficiency. This configuration allows T5 to capture nuanced linguistic patterns while remaining practical for real-world deployment. The vocabulary size of 32,128 tokens reflects SentencePiece's statistical optimization, providing broad coverage of multilingual text while maintaining manageable parameter counts. With over 60 million parameters, T5-small demonstrates that effective text-to-text modeling doesn't require enormous model sizes — the unified architecture efficiently shares knowledge across tasks, making each parameter contribute to multiple NLP capabilities simultaneously. This architectural efficiency is part of what makes T5 such a practical and versatile choice for diverse NLP applications.

---

### Conclusion and Next Steps

Congratulations on completing not just this lesson, but the entire Harnessing Transformers with Hugging Face course! Your dedication in reaching this final milestone demonstrates exceptional commitment to mastering one of the most transformative technologies in modern AI. From understanding attention mechanisms at their core to implementing complete transformer architectures, and finally to wielding state-of-the-art models through Hugging Face, you've traversed the complete landscape of transformer-based NLP. T5's elegant text-to-text unification serves as the perfect capstone, showing how architectural innovation can simplify complex problems while expanding capabilities.

As you prepare for the final practice section ahead, you carry with you a comprehensive toolkit: deep architectural understanding, practical implementation skills, and the ability to leverage cutting-edge models for real-world applications. Whether you're building the next breakthrough NLP application, contributing to open-source projects, or pushing the boundaries of language understanding, you now possess the foundation to make meaningful contributions to this rapidly evolving field. The transformers you've mastered here will continue to shape the future of AI, and you're now equipped to be part of that transformation!

## T5 Tokenizer Bug Hunt

Now that you've learned about T5's SentencePiece tokenization system, it's time to get hands-on experience with how it works in practice. You'll be working with a function that explores T5's tokenization, but something isn't working correctly.

---

### Your Task

The provided code contains a function called `explore_t5_tokenizer()` that should demonstrate how T5 breaks down text into tokens and converts them to token IDs. However, when you run it, you'll notice that the output doesn't match what you'd expect from SentencePiece tokenization.

Your job is to debug the code and fix the issues so that it properly shows how T5 tokenizes text like "Hello world" into tokens such as `['▁Hello', '▁world']`. Pay close attention to the method calls and tokenizer configuration — sometimes the smallest details make the biggest difference in getting tokenization to work correctly.

---

### Starter Code

```python
from transformers import T5ForConditionalGeneration, AutoTokenizer, pipeline
import torch

def explore_t5_tokenizer():
    """Explore T5's SentencePiece tokenization"""
    print("T5 SentencePiece Tokenization:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "Bonjour le monde"]

    for text in test_texts:
        tokens = tokenizer.encode(text)
        token_ids = tokenizer.tokenize(text)

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()

def main():
    explore_t5_tokenizer()

if __name__ == "__main__":
    main()
```

---

### Solution

This exercise has no `# TODO`. The bug is that the two method calls are swapped: `tokenizer.encode(text)` is assigned to `tokens` and `tokenizer.tokenize(text)` is assigned to `token_ids`. The fix is to swap them back.

```python
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)
```

- `tokenizer.tokenize(text)` returns the subword strings, such as `['▁Hello', '▁world']`. The `▁` marks the start of a word.
- `tokenizer.encode(text)` returns the integer vocabulary IDs, and T5 appends the end-of-sequence ID `1` at the end.
- With the calls swapped, the `Tokens:` line shows numbers and the `Token IDs:` line shows strings, so the labels describe the wrong data.
- Nothing crashes with the swap, since both methods accept a string. The bug only shows up in the printed output.
- The starter's `T5ForConditionalGeneration`, `pipeline` and `torch` imports are unused in this function. They are left as given.

When the script runs, each text prints its subword tokens and then its token IDs. For `Hello world`, the lesson shows `['▁Hello', '▁world']` and `[8774, 296, 1]`. The first run downloads the `t5-small` tokenizer files, so it needs internet access.

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer, pipeline
import torch

def explore_t5_tokenizer():
    """Explore T5's SentencePiece tokenization"""
    print("T5 SentencePiece Tokenization:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "Bonjour le monde"]

    for text in test_texts:
        # tokenize() gives subword strings, encode() gives token IDs
        tokens = tokenizer.tokenize(text)
        token_ids = tokenizer.encode(text)

        print(f"Text: '{text}'")
        print(f"  Tokens: {tokens}")
        print(f"  Token IDs: {token_ids}")
        print()

def main():
    explore_t5_tokenizer()

if __name__ == "__main__":
    main()


## Building Universal Text Processor

Now that you've mastered T5's tokenization mechanics, it's time to explore T5's unified text-to-text approach. Your objective is to complete the missing generation logic in the provided code, which demonstrates how T5 handles different NLP operations through prefixes.

---

### Your Task

The code structure includes model setup and a list of different operations with their prefixes and inputs. You need to implement the core functionality that:

- Combines prefixes with input text using proper formatting
- Encodes and processes the text through the model
- Generates and decodes the final results

This exercise will help you understand how T5 uses prefixes to guide its behavior across different NLP operations, showcasing the power of its unified framework.

---

### Starter Code

```python
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

def t5_text_to_text_tasks():
    """Demonstrate T5's text-to-text approach"""
    print("T5 Text-to-Text Tasks:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Different tasks with prefixes
    tasks = [
        {
            "prefix": "translate English to German:",
            "input": "Hello, how are you?",
            "task_name": "Translation"
        },
        {
            "prefix": "evaluate sentiment:",
            "input": "I loved this movie!",
            "task_name": "Sentiment analysis"
        }
    ]

    for task in tasks:
        # TODO: Combine the task prefix with the input text using f-string formatting
        input_text = task['input']

        # TODO: Encode the input text and generate model output
        # Hint: Use tokenizer.encode() with return_tensors="pt"
        # Then use model.generate() with appropriate parameters

        # TODO: Decode the output and print the results
        # Hint: Use tokenizer.decode() with skip_special_tokens=True

        print(f"Task: {task['task_name']}")
        print(f"  Input: {task['input']}")
        print(f"  Output: [Complete the generation logic]")
        print()

def main():
    t5_text_to_text_tasks()

if __name__ == "__main__":
    main()
```

---

### Solution

Three TODOs needed code: building the prefixed input, encoding and generating, and decoding the output into the printed result. The starter's `Output:` line is also a placeholder, so it now prints the decoded text.

```python
# Combine the task prefix with the input text
input_text = f"{task['prefix']} {task['input']}"

# Encode the input text and generate model output
input_ids = tokenizer.encode(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_length=30,
        num_beams=2,
        early_stopping=True
    )

# Decode the output and print the results
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"  Output: {output_text}")
```

- `f"{task['prefix']} {task['input']}"` joins the prefix and the input with one space, for example `translate English to German: Hello, how are you?`. The prefix is what tells T5 which task to run.
- `tokenizer.encode(input_text, return_tensors="pt")` turns the text into a PyTorch tensor of token IDs, shape `(1, sequence_length)`, which is what `generate` expects.
- `torch.no_grad()` turns off gradient tracking, since this is inference only.
- `model.generate(...)` produces the output token IDs. `max_length=30` caps the output length, `num_beams=2` keeps two candidate sequences at each step, and `early_stopping=True` stops once the beams finish.
- `tokenizer.decode(outputs[0], skip_special_tokens=True)` turns the first generated sequence back into text and removes special tokens such as `</s>`.
- Without the prefix, T5 gets no instruction and cannot tell which task is wanted.
- Without the decode step, `output_text` is undefined and the `Output:` line cannot print the result.

When the script runs, it prints one block per task. The lesson shows `Hallo, wie sind Sie?` for translation. For sentiment, `t5-small` gives the unclean `Evalu sentiment sentiment: I loved this movie!`, which is a model limit and not a bug in the code. The first run downloads the `t5-small` model, so it needs internet access.

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

def t5_text_to_text_tasks():
    """Demonstrate T5's text-to-text approach"""
    print("T5 Text-to-Text Tasks:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Different tasks with prefixes
    tasks = [
        {
            "prefix": "translate English to German:",
            "input": "Hello, how are you?",
            "task_name": "Translation"
        },
        {
            "prefix": "evaluate sentiment:",
            "input": "I loved this movie!",
            "task_name": "Sentiment analysis"
        }
    ]

    for task in tasks:
        # Combine the task prefix with the input text
        input_text = f"{task['prefix']} {task['input']}"

        # Encode the input text and generate model output
        input_ids = tokenizer.encode(input_text, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_length=30,
                num_beams=2,
                early_stopping=True
            )

        # Decode the output and print the results
        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"Task: {task['task_name']}")
        print(f"  Input: {task['input']}")
        print(f"  Output: {output_text}")
        print()

def main():
    t5_text_to_text_tasks()

if __name__ == "__main__":
    main()


## Perfecting T5 Text Generation

Now that you've worked with T5's text-to-text approach using prefixes, it's time to explore how different generation strategies affect the quality and consistency of T5's outputs.

---

### Your Task

Your objective is to compare the greedy decoding and beam search generation strategies in T5. The provided code attempts to demonstrate both approaches, but the generation calls are incomplete and will not produce the expected results.

You need to:

- Fix the greedy decoding generation to ensure deterministic output
- Configure beam search with exactly 3 beams and the proper parameters
- Make sure both strategies produce meaningful comparisons

Run the code first to see what happens, then debug the `t5_conditional_generation()` function to get both generation methods working correctly. Understanding these different decoding strategies will help you choose the right approach for your specific NLP applications.

---

### Starter Code

```python
from transformers import T5ForConditionalGeneration, AutoTokenizer, pipeline
import torch


def t5_conditional_generation():
    """Explore T5's conditional generation"""
    print("T5 Conditional Generation:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Test different generation strategies
    prompt = "translate English to French: I'm in love with Natural Language Processing!"
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    print(f"Prompt: '{prompt}'")

    with torch.no_grad():
        # TODO: Fix greedy decoding - add the missing parameter to ensure deterministic output
        greedy_output = model.generate(input_ids, max_length=20)
        greedy_text = tokenizer.decode(greedy_output[0], skip_special_tokens=True)

        # TODO: Fix beam search - set the correct number of beams to 3 and add missing parameter
        beam_output = model.generate(input_ids, max_length=20, num_beams=2)
        beam_text = tokenizer.decode(beam_output[0], skip_special_tokens=True)

    print(f"  Greedy: {greedy_text}")
    print(f"  Beam Search: {beam_text}")


def main():
    t5_conditional_generation()


if __name__ == "__main__":
    main()
```

---

### Solution

Two TODOs needed code: the greedy call needs `do_sample=False`, and the beam search call needs `num_beams=3` plus `do_sample=False`.

```python
# Greedy decoding: deterministic, always picks the most likely token
greedy_output = model.generate(input_ids, max_length=20, do_sample=False)

# Beam search: keep 3 candidate sequences, no sampling
beam_output = model.generate(input_ids, max_length=20, num_beams=3, do_sample=False)
```

- `do_sample=False` turns sampling off, so the model picks the highest-probability token at each step and the output is the same on every run. It is set explicitly so the choice is visible in the code.
- `num_beams=3` makes beam search track the 3 best partial sequences at each step and return the best finished one. The starter used `2`, and the task asks for exactly `3`.
- `do_sample=False` on the beam call keeps beam search deterministic. With `do_sample=True` it would become beam-search multinomial sampling, which is random.
- `max_length=20` is the same for both calls, so the two outputs are compared under the same length limit.
- Without `num_beams=3`, the beam call still runs but searches only 2 beams, so it does not match the task.
- Greedy decoding commits to the best token at each step. Beam search can find a sequence with a higher overall probability that a greedy choice would have missed.
- The starter's `pipeline` import is unused in this function. It is left as given.

When the script runs, it prints the prompt, then the greedy translation and the beam search translation. The lesson shows the same French sentence for both strategies on this prompt, `Je suis en amour avec le traitement des langues naturelles!`, so the two outputs may match. The first run downloads the `t5-small` model, so it needs internet access.

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer, pipeline
import torch


def t5_conditional_generation():
    """Explore T5's conditional generation"""
    print("T5 Conditional Generation:")

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Test different generation strategies
    prompt = "translate English to French: I'm in love with Natural Language Processing!"
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    print(f"Prompt: '{prompt}'")

    with torch.no_grad():
        # Greedy decoding: do_sample=False gives deterministic output
        greedy_output = model.generate(input_ids, max_length=20, do_sample=False)
        greedy_text = tokenizer.decode(greedy_output[0], skip_special_tokens=True)

        # Beam search: 3 beams, no sampling
        beam_output = model.generate(input_ids, max_length=20, num_beams=3, do_sample=False)
        beam_text = tokenizer.decode(beam_output[0], skip_special_tokens=True)

    print(f"  Greedy: {greedy_text}")
    print(f"  Beam Search: {beam_text}")


def main():
    t5_conditional_generation()


if __name__ == "__main__":
    main()


## T5 Model Architecture Explorer

## Building Your T5 Task Handler